 =====================================
# Step 1: Import Libraries
 =====================================

In [30]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime
from zoneinfo import ZoneInfo

 =====================================
# Step 2: Load Dataset
 =====================================

In [31]:
df = pd.read_csv(r"C:\Users\Vansh Sharma\Downloads\Play Store Data (1).csv")

df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


 =====================================
# Step 3: Clean Installs Column
 =====================================

In [32]:
df["Installs"] = (
    df["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

df["Installs"] = pd.to_numeric(df["Installs"], errors="coerce")

 =====================================
# Step 4: Clean Reviews Column & Filter (>500)
 =====================================

In [33]:
df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce")
df = df[df["Reviews"] > 500]

 =====================================
# Step 5: Filter App Names (x, y, z + no "S")
 =====================================

In [34]:
df = df[
    ~df["App"].str.startswith(("x", "y", "z"), na=False)
]

df = df[
    ~df["App"].str.contains("S", na=False)
]

 =====================================
# Step 6: Filter Categories (E, C, B only)
 =====================================

In [35]:
df = df[
    df["Category"].str.startswith(("E", "C", "B"), na=False)
]

 =====================================
# Step 7: Translate Categories
 =====================================

In [36]:
translation = {
    "BEAUTY": "सौंदर्य",   # Hindi
    "BUSINESS": "வணிகம்",  # Tamil
    "DATING": "Rencontres" # German
}

df["Category_Translated"] = df["Category"].replace(translation)

 =====================================
# Step 8: Create Monthly Time Column
 =====================================

In [37]:
df["Last Updated"] = pd.to_datetime(df["Last Updated"], errors="coerce")
df["Month"] = df["Last Updated"].dt.to_period("M").astype(str)

 =====================================
# Step 9: Create Monthly Trend Dataset
 =====================================

In [38]:
trend = df.groupby(["Month", "Category_Translated"])["Installs"].sum().reset_index()
trend = trend.sort_values("Month")
trend["Growth%"] = trend.groupby("Category_Translated")["Installs"].pct_change() * 100

 =====================================
# Step 10: Calculate date and time
 =====================================

In [39]:
ist = ZoneInfo("Asia/Kolkata")
now = datetime.now(ist).time()

start = datetime.strptime("18:00", "%H:%M").time()
end = datetime.strptime("21:00", "%H:%M").time()

 =====================================
# Step 11: Set Time Restriction (6 PM - 9 PM IST)
 =====================================

In [40]:
if start <= now <= end:

    fig = go.Figure()

    for cat in trend["Category_Translated"].unique():
        temp = trend[trend["Category_Translated"] == cat]

        # Line plot
        fig.add_trace(go.Scatter(
            x=temp["Month"],
            y=temp["Installs"],
            mode="lines+markers",
            name=cat
        ))

        # Highlight growth > 20%
        high = temp[temp["Growth%"] > 20]

        fig.add_trace(go.Scatter(
            x=high["Month"],
            y=high["Installs"],
            fill="tozeroy",
            mode="none",
            name=f"{cat} High Growth (>20%)",
            opacity=0.3
        ))

    fig.update_layout(
        title="Monthly Installs Growth with High Growth Highlight",
        template="plotly_white",
        xaxis_title="Month",
        yaxis_title="Installs"
    )

    fig.show()

else:
    print("Graph only available between 6 PM - 9 PM IST")


Graph only available between 6 PM - 9 PM IST


 =====================================
# Step 12: Build Plotly Visualization (Growth + Highlight)
 =====================================

In [41]:


import plotly.graph_objects as go

fig = go.Figure()

for cat in trend["Category_Translated"].unique():
    temp = trend[trend["Category_Translated"] == cat]

    # Line plot for installs over time
    fig.add_trace(go.Scatter(
        x=temp["Month"],
        y=temp["Installs"],
        mode="lines+markers",
        name=cat
    ))

    # Highlight areas where growth > 20%
    high = temp[temp["Growth%"] > 20]

    fig.add_trace(go.Scatter(
        x=high["Month"],
        y=high["Installs"],
        fill="tozeroy",
        mode="none",
        name=f"{cat} High Growth (>20%)",
        opacity=0.3
    ))

fig.update_layout(
    title="Monthly Installs Growth with High Growth Highlight",
    template="plotly_white",
    xaxis_title="Month",
    yaxis_title="Installs"
)